In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Pip installation and version update

!pip install pybamm
!pip uninstall numpy -y
!pip install numpy==1.23.5

In [ ]:
import pybamm
import numpy as np
import pandas as pd
import matplotlib as mpl
from cycler import cycler

In [ ]:
# Paramater and submodel search functions

model.variables.search("")
print(pybamm.lithium_ion.DFN().options)


{'calculate discharge energy': 'false', 'calculate heat source for isothermal models': 'false', 'cell geometry': 'arbitrary', 'contact resistance': 'false', 'convection': 'none', 'current collector': 'uniform', 'diffusivity': 'single', 'dimensionality': 0, 'electrolyte conductivity': 'default', 'exchange-current density': 'single', 'heat of mixing': 'false', 'hydrolysis': 'false', 'intercalation kinetics': 'symmetric Butler-Volmer', 'interface utilisation': 'full', 'lithium plating': 'none', 'lithium plating porosity change': 'false', 'loss of active material': 'none', 'number of MSMR reactions': 'none', 'open-circuit potential': 'single', 'operating mode': 'current', 'particle': 'Fickian diffusion', 'particle mechanics': 'none', 'particle phases': '1', 'particle shape': 'spherical', 'particle size': 'single', 'SEI': 'none', 'SEI film resistance': 'none', 'SEI on cracks': 'false', 'SEI porosity change': 'false', 'stress-induced diffusion': 'false', 'surface form': 'false', 'surface tem

In [ ]:
#1. LiB Simulation TEST Using Preset Parameters from PyBaMM

# Choose Doyle-Fuller-Newman (DFN) model for most accurate LiB representation or BaseModel (to set degradation variables)
model = pybamm.lithium_ion.DFN({"SEI": "reaction limited",               # SEI degradation
            "lithium plating": "reversible",         # Lithium plating degradation
            "particle mechanics": "swelling and cracking",
            "SEI film resistance": "distributed",
            "thermal": "lumped"  # Particle mechanics degradation
})

# Function to simulate battery degradation in space conditions
def generate_space_battery_data(temperatures, num_cycles=1, C_rate=25, cumulative_radiation_dose_Gy=0.0007): # Number of cycles per week where LEO batteries cycle ~5000 times a year (16 times per day)
    data_list = []                 # Positional argument                     # 5-19 Gy after 1100 days in LEO cubesats (16 cycles = 0.01 Gy)
    data_dict = {"Discharge Capacity (A.h)": {},
            "Capacity Fade (%) with Stresses": {},
            "Base Capacity Fade (%)": {},
            "State of Health (SOH) (%) with Stresses": {},
            "State of Health (SOH) (%)": {},
            "State of Power (SOP) (W)": {},
            "Loss of Lithium Inventory (LLI) (%)": {},
            "Loss of Lithium Inventory (LLI) (%) with Stresses": {}}

    for i,temp in enumerate(temperatures): # i refers to the first temperature at 273 K (temp 0) and allows for string
        print(f"Simulating at temperature: {temp} K")

        param = pybamm.ParameterValues("OKane2022")  # Example Preset Li-ion parameters can be included

        initial_state_of_charge = 0.1
        max_neg_concentration = param["Maximum concentration in negative electrode [mol.m-3]"]
        max_pos_concentration = param["Maximum concentration in positive electrode [mol.m-3]"]

# Defines SEI kinetic rate as affected by radiation
        k_R = 1.0e-3  # Radiation coefficient [3]
        base_sei_rate = 1e-14 * np.exp(0.325 * (temp - 298)) # SEI growth model without other stresses using Arrhenius temperature dependence at 24 kJ/mol Ea
        T_plating = 283  # Below 298 K where plating has the most affect
        k_LP = 0.01      # Empirical penalty per degree below threshold
        if temp < T_plating:
          plating_modifier = k_LP * (T_plating - temp)
        else:
          plating_modifier = 0
        k_T = 0.01  # Temperature coefficient
        temp_delta = temp - 298  # Difference from STP reference
        radiation_modifier = k_R * cumulative_radiation_dose_Gy # 1 Gy increases degradation by 0.2%
        temperature_modifier = k_T * temp_delta # 1 K increase above 298 K increases degradation by 1% and vice versa
        modifier_total = 1 + temperature_modifier + radiation_modifier + plating_modifier # A multiplicative multiplier that is the sum of modifiers that represent deviations from 0 Gy, 298 K, and 760 Torr
        sei_rate_mod = base_sei_rate * modifier_total # SEI growth model with stresses

# Manually set basic battery performance parameters
        param.update({
            "Upper voltage cut-off [V]": 4.05, # SoC limits for individual cells in a pack
            "Lower voltage cut-off [V]": 2.50,
            "Nominal cell capacity [A.h]": 25.0,
            "Ambient temperature [K]": temp,
            "SEI kinetic rate constant [m.s-1]": sei_rate_mod,
# Sets initial charge state to be lower than default Chen2020 parameters
            "Initial concentration in negative electrode [mol.m-3]": initial_state_of_charge * max_neg_concentration,
            "Initial concentration in positive electrode [mol.m-3]": max_pos_concentration * (1-initial_state_of_charge),
            "Positive electrode thickness [m]":
              param["Positive electrode thickness [m]"] * (25.0 / 4.85),
            "Negative electrode thickness [m]":
              param["Negative electrode thickness [m]"] * (25.0 / 4.85),
            "Electrode width [m]":
              param["Electrode width [m]"] * (25.0 / 4.85)
            })

# Defines realistic charge-discharge cycle for an LEO spaceraft (90 minute orbit, 55 minutes of charge and 35 minutes of discharge)
        experiment = pybamm.Experiment(
            [ #  Lower charge and discharge rates to remedy negative y-axis values on multiple plots graphs
                "Charge at 12.5 A until 4.05 V", # Divided the 12.5 and 17.14 A for charge and discharge by the number of cells (8)
                "Hold at 4.05 V for 55 minutes",
                "Discharge at 17.14 A until 2.5 V" # Comment out discharge to see generate single temperature plots with only charge or vice versa
            ] * num_cycles         # until 40% DoD
        )

# Simulates the experiment over a duration
        sim = pybamm.Simulation(model, experiment=experiment, parameter_values=param) # solver=pybamm.CasadiSolver(mode="safe")
        sim.solve()

# Generates data set keys
        discharge_capacity_ah_entries = sim.solution["Discharge capacity [A.h]"].entries
        discharge_capacity_ah = discharge_capacity_ah_entries[-1]
        terminal_voltage = sim.solution["Terminal voltage [V]"].entries[-1] # EOD is 3.6 V for one cell
        sei_thickness_m = sim.solution["X-averaged negative total SEI thickness [m]"].entries[-1]
        neg_crack_length_m = sim.solution["X-averaged negative particle crack length [m]"].entries[-1]
        pos_crack_length_m = sim.solution["X-averaged positive particle crack length [m]"].entries[-1]
        charge_rate_A = 12.5 # Amps/current for charge and discharge for one cell in a pack of 8
        discharge_rate_A = 17.5
        t_rest_s = 0   # From experiment definiton
        t_charge_s = 3300
        t_discharge_s = 2100
        res_ohm_entries = np.atleast_1d(sim.solution["Resistance [Ohm]"].entries[-1])
        res_ohm = res_ohm_entries
        ecm_res_ohm = sim.solution["Local ECM resistance [Ohm]"].entries[-1]
        rated_capacity_ah = 25  # Also called nameplate/nominal capacity, in this case it is the same for all cells since they are connected in a series
        sop_watts_entries = [(terminal_voltage ** 2) / (4 * element) for element in list(res_ohm_entries)]
        sop_watts = sop_watts_entries[-1]
        max_voltage = 4.05 # From manually set parameters
        min_voltage = 2.50
        pressure_torr = 10e-10
        gravity = 8.82 # m/s^2

        m = 0.0373  # Ah per K (based on ~4% capacity gain per 10 K at 0.7C) 1 cycle = 0.0373, 16 cycles = 0.08145
        Q_ref_list = [m * (temp - 296) + 10 for temp in temperatures]

        base_capacity_fade_entries = [(Q_ref - element) / Q_ref for element in list(discharge_capacity_ah_entries)
          for Q_ref in Q_ref_list]
        base_capacity_fade = ((m * (temp - 296) + 10) - discharge_capacity_ah) / (m * (temp - 296) + 10) # 25 Ah (rated capacity) is too large to compare to discharge capacity for one cell
        capacity_fade_mod_entries = [100 * element * modifier_total for element in list(base_capacity_fade_entries)]
        capacity_fade_mod = 100 * base_capacity_fade * modifier_total
        soh_percent_entries = [100 - (element * 100) for element in list(base_capacity_fade_entries)]
        soh_percent = 100 - (base_capacity_fade * 100)
        soh_mod_entries = [100 - element for element in list(capacity_fade_mod_entries)]
        soh_mod = 100 - capacity_fade_mod
        lli_percent_entries = sim.solution["Loss of lithium inventory [%]"].entries
        lli_percent = lli_percent_entries[-1]
        lli_percent_mod_entries = [element * modifier_total for element in list(lli_percent_entries)]
        lli_percent_mod = lli_percent * modifier_total  # SEI μm x mod

# Anomaly Detection and Flagging/RUL health classification
        def classify_degradation(rul_deviation): # Health flag based on deviation percent
          if abs(rul_deviation) < 10:
            return "green"
          elif abs(rul_deviation) < 20:
            return "amber"
          else:
            return "red"

        fade_per_cycle = (1 - (discharge_capacity_ah / rated_capacity_ah)) / num_cycles
        fade_per_cycle_mod = ((1 - (discharge_capacity_ah / rated_capacity_ah)) * modifier_total) / num_cycles
        if fade_per_cycle > 0 and fade_per_cycle_mod > 0:
          est_rul_cycles = (0.2) / fade_per_cycle  # 20% fade = EOL, estimated remaining cycles to EOL
          est_rul_cycles_mod = (0.2) / fade_per_cycle_mod
        else:
          est_rul_cycles = float('inf')
          est_rul_cycles_mod = float('inf')
        rul_deviation = 100 * (1 - (est_rul_cycles_mod / est_rul_cycles)) # Deviation between modified RUL and baseline RUL
        flag_color = classify_degradation(rul_deviation)

        data_list.append({
            "Temperature (K)": temp,
            "Discharge Capacity (A.h)": discharge_capacity_ah,
            #"SEI Thickness (m)": sei_thickness_m, #
            #"Negative Particle Crack Length (m)": neg_crack_length_m, #
            #"Positive Particle Crack Length (m)": pos_crack_length_m, #
            #"Cycle Index": num_cycles,
            #"Minimum Voltage (V)": min_voltage,
            #"Maximum Voltage (V)": max_voltage,
            #"Charge Rate (A)": charge_rate_A,
            #"Discharge Rate (A)": discharge_rate_A,
            #"Rest Time (s)": t_rest_s,
            #"Charge Time (s)": t_charge_s,
            #"Discharge Time (s)": t_discharge_s,
            "Resistance (Ohm)": res_ohm,
            "Local ECM Resistance (Ohm)": ecm_res_ohm,
            "Capacity Fade (%) with Stresses": capacity_fade_mod, # In space conditions
            "Base Capacity Fade (%)": base_capacity_fade * 100,
            "State of Health (SOH) (%) with Stresses": soh_mod, # In space conditions
            "State of Health (SOH) (%)": soh_percent,
            "State of Power (SOP) (W)": sop_watts,
            "Loss of Lithium Inventory (LLI) (%)": lli_percent,
            "Loss of Lithium Inventory (LLI) (%) with Stresses": lli_percent_mod, # In space conditions
            #"Radiation Dose (Gy)": cumulative_radiation_dose_Gy,
            "SEI Growth (m.s-1)": base_sei_rate,
            "SEI Growth with Stresses (m.s-1)": sei_rate_mod,

            #"Estimated RUL (cycles)": rul_cycles,
            #"RUL Deviation (%)": rul_deviation_pct,
            #"Health Flag": flag_color,

            "Degradation Severity": classify_degradation(rul_deviation),
            "Estimated RUL (Base, cycles)": est_rul_cycles,
            "Estimated RUL (Mod, cycles)": est_rul_cycles_mod,
            "RUL Deviation (%)": rul_deviation,

            #"Pressure": pressure_torr,
            #"Gravity": gravity
                })

        data_dict["Discharge Capacity (A.h)"][f"{round(temp)} K"] = discharge_capacity_ah_entries
        data_dict["Capacity Fade (%) with Stresses"][f"{round(temp)} K"] = capacity_fade_mod_entries
        data_dict["Base Capacity Fade (%)"][f"{round(temp)} K"] = base_capacity_fade_entries
        data_dict["State of Health (SOH) (%) with Stresses"][f"{round(temp)} K"] = soh_mod_entries
        data_dict["State of Health (SOH) (%)"][f"{round(temp)} K"] = soh_percent_entries
        data_dict["State of Power (SOP) (W)"][f"{round(temp)} K"] = sop_watts_entries
        data_dict["Loss of Lithium Inventory (LLI) (%)"][f"{round(temp)} K"] = lli_percent_entries
        data_dict["Loss of Lithium Inventory (LLI) (%) with Stresses"][f"{round(temp)} K"] = lli_percent_mod_entries # .entries generates the time series array for the full experiment

    df_results = pd.DataFrame(data_list)
   df_results.to_csv('space_battery_degradation_dataset.csv', index=False)
    return df_results, sim, data_dict


In [ ]:
# Prints the full table of df_results for each set of temperature values, this df_results can also be removed to print only an array of data_dict values

training_temps = np.linspace(273, 293, 10) # Realistic spacecraft temp range for LEO (0°C to 20°C)
testing_temps = np.linspace(293, 313, 10) # (20°C to 40°C)

df_results_training, sim_training, data_dict_training = generate_space_battery_data(temperatures=training_temps)
df_results_test, sim_test, data_dict_test = generate_space_battery_data(temperatures=testing_temps)
df_results_training


Simulating at temperature: 273.0 K
Simulating at temperature: 275.22222222222223 K
Simulating at temperature: 277.44444444444446 K
Simulating at temperature: 279.6666666666667 K
Simulating at temperature: 281.8888888888889 K
Simulating at temperature: 284.1111111111111 K
Simulating at temperature: 286.3333333333333 K
Simulating at temperature: 288.55555555555554 K
Simulating at temperature: 290.77777777777777 K
Simulating at temperature: 293.0 K


In [ ]:
# Defines figure functions in LaTeX format for the data set defined above

mpl.rcParams.update({
    "text.usetex": False,                  # change to True if LaTeX is available
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Nimbus Roman", "DejaVu Serif"],
    "figure.figsize": (3.8, 2.7),
    "figure.dpi": 300,
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.linewidth": 0.4,
    "grid.alpha": 0.3,
    "axes.prop_cycle": cycler("color", mpl.colormaps["tab10"].colors),
    "lines.linewidth": 1.6,
    "lines.markersize": 4,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.minor.size": 2,
    "ytick.minor.size": 2,
    "axes.labelsize": 10,
    "font.size": 9,
    "axes.titlesize": 10,
})

mpl.rcParams.update({
    "legend.fontsize": 8,
    "legend.handlelength": 1.4,
    "legend.markerscale": 0.8,
    "legend.frameon": True,
    "legend.labelspacing": 0.2,
})

import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from matplotlib.ticker import MaxNLocator

def plot_function(x_values, y_values, x_axis_label, y_axis_label, plot_name, save_dir="/content/drive/MyDrive/PyBaMM Plots v.3"):
  plt.gca().yaxis.set_major_formatter(FormatStrFormatter('%.0f'))
  plt.gca().yaxis.set_major_locator(MaxNLocator(integer=True))
  plt.figure()
  plt.plot(x_values, y_values, marker = "o", markevery = max(len(x_values)//20, 1)) # Marks every n amount of points
  plt.xlabel(x_axis_label)
  plt.ylabel(y_axis_label)
  plt.minorticks_on()
  plt.tight_layout(pad=0.6)
  plt.savefig(f"{save_dir}/{plot_name}.jpg")

# Defines x and y values as arguments in the plots
temp_values = df_results_training["Temperature (K)"]

discharge_values_train = df_results_training["Discharge Capacity (A.h)"]
discharge_values_test = df_results_test["Discharge Capacity (A.h)"]

soh_values_mod = df_results_training["State of Health (SOH) (%) with Stresses"]
capacity_fade_values_mod = df_results_training["Capacity Fade (%) with Stresses"]
lli_values_mod = df_results_training["Loss of Lithium Inventory (LLI) (%) with Stresses"]
soh_values = df_results_training["State of Health (SOH) (%)"]
capacity_fade_values = df_results_training["Base Capacity Fade (%)"]
lli_values = df_results_training["Loss of Lithium Inventory (LLI) (%)"]
sop_values = df_results_training["State of Power (SOP) (W)"]
res_ohm_values = df_results_training["Resistance (Ohm)"]

# Individual plots for the range of temperatures
plot_function(temp_values, discharge_values_train, "Temperature (K)", "Discharge Capacity (A.h)", "Discharge_vs_Temp")
plot_function(temp_values, soh_values, "Temperature (K)", "State of Health (SOH) (%)", "SoH_vs_Temp")
plot_function(temp_values, capacity_fade_values, "Temperature (K)", "Base Capacity Fade (%)", "Base_Capacity_fade_vs_Temp")
plot_function(temp_values, lli_values, "Temperature (K)", "Loss of Lithium Inventory (LLI) (%)", "LLI_vs_Temp")
plot_function(temp_values, sop_values, "Temperature (K)", "State of Power (SOP) (W)", "SOP_vs_Temp")
plot_function(temp_values, capacity_fade_values_mod, "Temperature (K)", "Capacity Fade (%)", "Capacity_Fade_Degradation_vs_Temp")
plot_function(temp_values, soh_values_mod, "Temperature (K)", "State of Health (SOH) (%)", "SOH_Degradation_vs_Temp")
plot_function(temp_values, lli_values_mod, "Temperature (K)", "Loss of Lithium Inventory (LLI) (%)", "LLI_Degradation_vs_Temp")

# Multiple plots for that indicate the behavior at each temperature for one cycle
def multiple_plot_function(y_axis_label, data_dict, plot_name, save_dir="/content/drive/MyDrive/PyBaMM Plots v.3"):
  plt.figure()
  for temp in data_dict[y_axis_label].keys():
#   print(len(np.arange(len(data_dict[key]))), len(data[key])) Length of values per temperature starting at 273 K
    plt.plot(np.arange(len(data_dict[y_axis_label][temp])), data_dict[y_axis_label][temp], label = f"{temp}")
    plt.xlabel("Time (s)")
    plt.ylabel(y_axis_label)
    plt.minorticks_on()
    plt.legend()
    plt.tight_layout(pad=0.2)
    plt.savefig(f"{save_dir}/{plot_name}.jpg")

multiple_plot_function("State of Health (SOH) (%)", data_dict_training, "SOH for All Temperatures")
multiple_plot_function("Discharge Capacity (A.h)", data_dict_training, "Discharge Capacity for All Temperatures")
multiple_plot_function("Loss of Lithium Inventory (LLI) (%)", data_dict_training, "LLI for All Temperatures")
multiple_plot_function("Base Capacity Fade (%)", data_dict_training, "Base Capacity Fade for All Tememperatures")


In [ ]:
# LR Model (for interpolation and extrapolation)

from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error

training_temp_array = np.array(training_temps.tolist()).reshape(-1, 1)
testing_temp_array = np.array(testing_temps.tolist()).reshape(-1, 1)

training_discharge_list = discharge_values_train.tolist()
training_discharge_array = np.array(training_discharge_list).reshape(-1, 1) #
test_discharge_list = discharge_values_test.tolist()
test_discharge_array = np.array(test_discharge_list).reshape(-1, 1)

soh_list = soh_values.tolist()
soh_array = np.array(soh_list).reshape(-1, 1)
capacity_fade_list = capacity_fade_values.tolist()
capacity_fade_array = np.array(capacity_fade_list).reshape(-1, 1)
lli_list = lli_values.tolist()
lli_array = np.array(lli_list).reshape(-1, 1)

# Array for that combines all training and testing temperatures
all_discharge_array = np.concatenate([training_discharge_array, test_discharge_array])
all_temp_array = np.concatenate([training_temp_array, testing_temp_array])

# Splits arrays by the last three values for optimal extrapolation
training_temp_array, training_discharge_array = all_temp_array[:-10], all_discharge_array[:-10]
testing_temp_array, test_discharge_array = all_temp_array[-10:], all_discharge_array[-10:]

LR = LinearRegression()
LR.fit(training_temp_array, training_discharge_array) # First turn into lists and then arrays when np.array error arises

print("Coefficient (weight):", LR.coef_)
print("y Intercept:", LR.intercept_)
print("Predicted discharge capacity:", LR.predict(np.array([[313]]))[0][0]) # Or testing temps for lowest and highest temperature values

# Evaluates the model score (R²), where the best value is 1.0
model_score_train = LR.score(training_temp_array, training_discharge_array, sample_weight=None)
print("Model R² score (Training):", model_score_train)
model_score_test = LR.score(testing_temp_array, test_discharge_array, sample_weight=None)
print("Model R² score (Testing):", model_score_test)

# Calculates the Mean Squared Error (MSE)
y_train_pred = LR.predict(training_temp_array)
mse_train = mean_squared_error(training_discharge_array, y_train_pred)
print("Mean Squared Training Error (MSE):", mse_train)

y_test_pred = LR.predict(testing_temp_array)
mse_test = mean_squared_error(test_discharge_array, y_test_pred)
print("Mean Squared Testing Error (MSE):", mse_test)


Coefficient (weight): [[0.00747833]]
y Intercept: [7.10210599]
Predicted discharge capacity: 9.442822020953303
Model R² score (Training): 0.9961887165484326
Model R² score (Testing): -0.4542334228396321
Mean Squared Training Error (MSE): 8.717005802450427e-06
Mean Squared Testing Error (MSE): 0.0012391706847825162


In [ ]:
# RF Model (for interpolation)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Shuffle and split from all temperatures for most accurate MSE and R² (75% train, 25% test)
training_temp_array, testing_temp_array, training_discharge_array, test_discharge_array = train_test_split(
    all_temp_array, all_discharge_array, test_size=0.25, random_state=0
)


RF = RandomForestRegressor(max_depth=3, random_state=0, min_samples_split = 2)
RF.fit(training_temp_array, training_discharge_array)

# Predicts discharge for a set temperature
#predicted_discharge = RF.predict([[313]])
#print("Predicted discharge capacity:", predicted_discharge)

# Evaluates the model score (R²), where the best value is 1.0
print("Train R²:", RF.score(training_temp_array, training_discharge_array))
print("Test R²:", RF.score(testing_temp_array, test_discharge_array))

# Calculates the Mean Squared Error (MSE)
y_train_pred = RF.predict(training_temp_array)
mse_train = mean_squared_error(training_discharge_array, y_train_pred)
print("Mean Squared Training Error (MSE):", mse_train)

y_test_pred = RF.predict(testing_temp_array)
mse_test = mean_squared_error(test_discharge_array, y_test_pred)
print("Mean Squared Testing Error (MSE):", mse_test)

Train R²: 0.9924946602902547
Test R²: 0.9588862463665653
Mean Squared Training Error (MSE): 6.0932941961262895e-05
Mean Squared Testing Error (MSE): 0.0004961420293822901


In [ ]:
# Figures for LR and RF models

def plot_LR_model(x_values, y_values, model_data, x_axis_label, y_axis_label, plot_name, data_type, save_dir="/content/drive/MyDrive/PyBaMM Plots v.3"): # Slope intercept
  plt.figure()
  plt.scatter(x_values, y_values, label=f"{data_type} Data") # or testing data
  plt.xlabel(x_axis_label)
  plt.ylabel(y_axis_label)
  plt.plot(x_values, np.array([model_data["Slope"] * element + model_data["y Intercept"] for element in x_values]), label="Linear Regression Model", color="green")
  plt.legend()
  plt.tight_layout(pad=0.4)
  #print(x_values)
  plt.savefig(f"{save_dir}/{plot_name}.jpg")

#print(training_temp_array, training_discharge_array)
model_data = {"Slope": LR.coef_[0][0], "y Intercept": LR.intercept_[0]}
plot_LR_model(testing_temp_array, test_discharge_array, model_data, "Testing Temperatures (K)", "Discharge Capacity (A.h)", "LR_Slope_Test", "Testing")
plot_LR_model(training_temp_array, training_discharge_array, model_data, "Training Temperatures (K)", "Discharge Capacity (A.h)", "LR_Slope_Train", "Training")


def plot_RF_model(x_values, y_values, x_axis_label, y_axis_label, plot_name, data_type, save_dir="/content/drive/MyDrive/PyBaMM Plots v.3"): # Slope intercept
  plt.figure()
  plt.scatter(x_values, y_values, label=f"{data_type} Data") # or testing data
  plt.xlabel(x_axis_label)
  plt.ylabel(y_axis_label)
  plt.scatter(x_values, RF.predict(x_values), label="Random Forest Model", color="orange")
  plt.legend()
  plt.tight_layout(pad=0.4)
  #plt.savefig(f"{save_dir}/{plot_name}.jpg")

#print(RF.predict(training_temp_array))
#plot_RF_model(testing_temp_array, test_discharge_array, "Testing Temperatures (K)", "Discharge Capacity (A.h)", "RF_Slope_Test", "Testing")
#plot_RF_model(training_temp_array, training_discharge_array, "Training Temperatures (K)", "Discharge Capacity (A.h)", "RF_Slope_Train", "Training")
